In [ ]:
import sys
import cv2
import numpy as np
import pandas as pd
from pathlib import Path

from skimage.feature import local_binary_pattern, graycomatrix, graycoprops
from skimage.filters import gabor_kernel
from skimage.measure import shannon_entropy
from scipy.stats import skew, kurtosis
from sklearn.preprocessing import StandardScaler

sys.path.append(str(Path.cwd().parent))
from src.segmentation.edge_detection import sobel_gauss_otsu, bordes_sv, densidad_bordes
from src.segmentation.morphological_ops import rellenar_contornos

PROCESSED_DIR = Path('../data/processed/train')
EXT = {'.jpg', '.jpeg', '.png'}

In [ ]:
UMBRAL_DENSIDAD = 0.15
K_GAUSS = 11

# textura clasica
LBP_P = 8
LBP_R = 1
LBP_BINS = LBP_P + 2
GLCM_DIST = [1]
GLCM_ANG = [0, np.pi/4, np.pi/2, 3*np.pi/4]
GLCM_PROPS = ['contrast', 'homogeneity', 'energy', 'correlation']

# gabor
GABOR_FREQS = [0.1, 0.2, 0.3]
GABOR_THETAS = [0, np.pi/4, np.pi/2, 3*np.pi/4]
GABOR_KERNELS = [(f, t, np.real(gabor_kernel(f, theta=t)))
                 for f in GABOR_FREQS for t in GABOR_THETAS]

CLASES = ['battery', 'cardboard', 'glass', 'metal', 'organic', 'paper', 'plastic', 'textile', 'trash']
N_IMGS = 100

In [ ]:
def features_textura(gray, mask=None):
    f = {}
    if mask is not None:
        ys, xs = np.where(mask > 0)
        if len(xs) == 0:
            f.update({f'lbp_{i}': 0 for i in range(LBP_BINS)})
            f.update({f'glcm_{p}': 0 for p in GLCM_PROPS})
            return f
        y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()
        region = gray[y0:y1 + 1, x0:x1 + 1]
    else:
        region = gray
    lbp = local_binary_pattern(region, LBP_P, LBP_R, method='uniform')
    hist, _ = np.histogram(lbp.ravel(), bins=LBP_BINS, range=(0, LBP_BINS), density=True)
    for i, h in enumerate(hist):
        f[f'lbp_{i}'] = h
    glcm = graycomatrix(region, GLCM_DIST, GLCM_ANG, levels=256, symmetric=True, normed=True)
    for prop in GLCM_PROPS:
        f[f'glcm_{prop}'] = graycoprops(glcm, prop).mean()
    return f


def features_gabor(gray, mask=None):
    f = {}
    if mask is not None:
        ys, xs = np.where(mask > 0)
        if len(xs) == 0:
            for i in range(len(GABOR_KERNELS)):
                f[f'gabor_{i}_mean'] = 0
                f[f'gabor_{i}_std'] = 0
            return f
        y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()
        region = gray[y0:y1 + 1, x0:x1 + 1].astype(np.float64)
    else:
        region = gray.astype(np.float64)
    for i, (freq, theta, kernel) in enumerate(GABOR_KERNELS):
        filtrado = cv2.filter2D(region, cv2.CV_64F, kernel)
        f[f'gabor_{i}_mean'] = filtrado.mean()
        f[f'gabor_{i}_std'] = filtrado.std()
    return f


def features_intensidad(gray, mask=None):
    f = {}
    vals = gray[mask > 0].astype(np.float64) if mask is not None else gray.ravel().astype(np.float64)
    if vals.size == 0:
        return {k: 0 for k in ['int_mean', 'int_std', 'int_skew', 'int_kurt', 'int_entropia']}
    f['int_mean'] = vals.mean()
    f['int_std'] = vals.std()
    f['int_skew'] = skew(vals)
    f['int_kurt'] = kurtosis(vals)
    f['int_entropia'] = shannon_entropy(vals)
    return f


def features_forma(mask, img_h, img_w):
    f = {}
    contornos, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contornos:
        return {k: 0 for k in ['area', 'perimetro', 'extent', 'solidity', 'aspect_ratio', 'circularidad']}
    c = max(contornos, key=cv2.contourArea)
    area = cv2.contourArea(c)
    perim = cv2.arcLength(c, True)
    x, y, w, h = cv2.boundingRect(c)
    hull = cv2.convexHull(c)
    area_hull = cv2.contourArea(hull)
    f['area'] = area / (img_h * img_w)
    f['perimetro'] = perim / (2 * (img_h + img_w))
    f['extent'] = area / (w * h) if w * h > 0 else 0
    f['solidity'] = area / area_hull if area_hull > 0 else 0
    f['aspect_ratio'] = w / h if h > 0 else 0
    f['circularidad'] = 4 * np.pi * area / (perim ** 2) if perim > 0 else 0
    return f


def features_hu(mask):
    f = {}
    hu = cv2.HuMoments(cv2.moments(mask)).flatten()
    for i, h in enumerate(hu):
        f[f'hu_{i}'] = -np.sign(h) * np.log10(abs(h) + 1e-12)
    return f


def features_color(hsv, mask):
    f = {}
    for i, ch in enumerate(['h', 's', 'v']):
        vals = hsv[:, :, i][mask > 0]
        f[f'color_{ch}_mean'] = vals.mean() if vals.size else 0
        f[f'color_{ch}_std'] = vals.std() if vals.size else 0
    return f


def features_color_extra(img_bgr, mask):
    f = {}
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    for i, ch in enumerate(['h', 's', 'v']):
        vals = hsv[:, :, i][mask > 0]
        f[f'color_{ch}_med'] = np.median(vals) if vals.size else 0
        f[f'color_{ch}_p25'] = np.percentile(vals, 25) if vals.size else 0
        f[f'color_{ch}_p75'] = np.percentile(vals, 75) if vals.size else 0
    for i, ch in zip([1, 2], ['a', 'b']):
        vals = lab[:, :, i][mask > 0]
        f[f'lab_{ch}_mean'] = vals.mean() if vals.size else 0
        f[f'lab_{ch}_std'] = vals.std() if vals.size else 0
    return f

In [ ]:
def procesar_imagen(img_path, clase):
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        return None
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    img_h, img_w = gray.shape

    hsv_raw = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    S_raw = cv2.GaussianBlur(hsv_raw[:, :, 1], (5, 5), 0)
    densidad = densidad_bordes(S_raw, K_GAUSS)
    img_lp = cv2.bilateralFilter(img_bgr, 50, 220, 300) if densidad > UMBRAL_DENSIDAD else img_bgr.copy()

    img_hsv = cv2.cvtColor(img_lp, cv2.COLOR_BGR2HSV)
    H, S, V = cv2.split(img_hsv)
    sobel_comb = bordes_sv(S, V, K_GAUSS)
    _, _, relleno, _, _ = rellenar_contornos(sobel_comb)

    fila = {'archivo': img_path.name, 'clase': clase}
    for k, v in features_textura(gray, relleno).items():
        fila[f'obj_{k}'] = v
    for k, v in features_textura(gray, None).items():
        fila[f'full_{k}'] = v
    for k, v in features_gabor(gray, relleno).items():
        fila[f'obj_{k}'] = v
    for k, v in features_gabor(gray, None).items():
        fila[f'full_{k}'] = v
    for k, v in features_intensidad(gray, relleno).items():
        fila[k] = v
    for k, v in features_forma(relleno, img_h, img_w).items():
        fila[f'forma_{k}'] = v
    for k, v in features_hu(relleno).items():
        fila[f'forma_{k}'] = v
    for k, v in features_color(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV), relleno).items():
        fila[k] = v
    for k, v in features_color_extra(img_bgr, relleno).items():
        fila[k] = v
    return fila

In [ ]:
split = 'train'
carpeta_split = Path('../data/processed') / split
FEATURES_DIR = Path('../results/features')
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

filas = []
for clase in CLASES:
    carpeta = carpeta_split / clase
    imagenes = sorted(p for p in carpeta.iterdir() if p.suffix.lower() in EXT)
    for j, img_path in enumerate(imagenes):
        fila = procesar_imagen(img_path, clase)
        if fila is not None:
            filas.append(fila)
        if (j + 1) % 100 == 0:
            print(f'  {clase}: {j + 1}/{len(imagenes)}')
    print(f'{clase}: listo')

df_train = pd.DataFrame(filas)
df_train.to_csv(FEATURES_DIR / 'features_train.csv', index=False)
print(f'>>> Guardado features_train.csv ({len(df_train)} filas)')

In [ ]:
split = 'val'
carpeta_split = Path('../data/processed') / split
FEATURES_DIR = Path('../results/features')
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

filas = []
for clase in CLASES:
    carpeta = carpeta_split / clase
    imagenes = sorted(p for p in carpeta.iterdir() if p.suffix.lower() in EXT)
    for j, img_path in enumerate(imagenes):
        fila = procesar_imagen(img_path, clase)
        if fila is not None:
            filas.append(fila)
        if (j + 1) % 100 == 0:
            print(f'  {clase}: {j + 1}/{len(imagenes)}')
    print(f'{clase}: listo')

df_val = pd.DataFrame(filas)
df_val.to_csv(FEATURES_DIR / 'features_val.csv', index=False)
print(f'>>> Guardado features_val.csv ({len(df_val)} filas)')

In [ ]:
split = 'test'
carpeta_split = Path('../data/processed') / split
FEATURES_DIR = Path('../results/features')
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

filas = []
for clase in CLASES:
    carpeta = carpeta_split / clase
    imagenes = sorted(p for p in carpeta.iterdir() if p.suffix.lower() in EXT)
    for j, img_path in enumerate(imagenes):
        fila = procesar_imagen(img_path, clase)
        if fila is not None:
            filas.append(fila)
        if (j + 1) % 100 == 0:
            print(f'  {clase}: {j + 1}/{len(imagenes)}')
    print(f'{clase}: listo')

df_test = pd.DataFrame(filas)
df_test.to_csv(FEATURES_DIR / 'features_test.csv', index=False)
print(f'>>> Guardado features_test.csv ({len(df_test)} filas)')

In [ ]:
num = df_feat.select_dtypes('number').replace([np.inf, -np.inf], np.nan).fillna(0)
scaled = pd.DataFrame(StandardScaler().fit_transform(num), columns=num.columns)
scaled['clase'] = df_feat['clase'].values

resumen_z = scaled.groupby('clase').mean(numeric_only=True).T
spread_z = resumen_z.std(axis=1).sort_values(ascending=False)
print('Top 20 separadores:')
print(spread_z.head(20))

In [ ]:
FEATURES_DIR = Path('../results/features')
FEATURES_DIR.mkdir(parents=True, exist_ok=True)
df_feat.to_csv(FEATURES_DIR / 'features_exploracion_9clases.csv', index=False)
print(f'Guardado ({len(df_feat)} filas)')